# QASM → stim: Deutsch–Jozsa on 16 qubits at $d = 5$

One worked example, end to end: a Deutsch–Jozsa circuit (16 qubits,
balanced oracle) compiles to a runnable stim circuit at distance 5,
the compiled circuit passes the verification checks, and its logical
error rate is measured under uniform circuit-level depolarizing noise
at $p = 10^{-3}$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from circls import compile_qasm, verify, measure_ler

In [ ]:
DJ16 = """OPENQASM 2.0;
include "qelib1.inc";
qreg q[16];
x q[15];
h q[0];
h q[1];
h q[2];
h q[3];
h q[4];
h q[5];
h q[6];
h q[7];
h q[8];
h q[9];
h q[10];
h q[11];
h q[12];
h q[13];
h q[14];
h q[15];
cx q[0],q[15];
cx q[1],q[15];
cx q[2],q[15];
cx q[3],q[15];
cx q[4],q[15];
cx q[5],q[15];
cx q[6],q[15];
cx q[7],q[15];
cx q[8],q[15];
cx q[9],q[15];
cx q[10],q[15];
cx q[11],q[15];
cx q[12],q[15];
cx q[13],q[15];
cx q[14],q[15];
h q[0];
h q[1];
h q[2];
h q[3];
h q[4];
h q[5];
h q[6];
h q[7];
h q[8];
h q[9];
h q[10];
h q[11];
h q[12];
h q[13];
h q[14];
creg c[16];
measure q -> c;"""

## Compile at distance 5

`compile_qasm` lowers the QASM through the full pipeline: the
front-end turns it into a PPM sequence, re-selection rewrites the
terminal measurement set, the PPMs are ordered and mapped, every
joint measurement gets a corridor, and the construction rules emit
the stim circuit.  Compilation takes about two minutes at
$d = 5$.

In [ ]:
out = compile_qasm(DJ16, distance=5)   # takes about two minutes
stats = out.stats()
print("patches           :", len(out.placement))
print("joint measurements:", sum(1 for r in out.routes if r))
print("measurement layers:", stats["T1_rounds"])
print("allocated volume  :", round(stats["V1_volume_blocks"], 1), "blocks")

## Verify

No detector fires at $p = 0$, the program observables are
deterministic, and the program bits carry the same affine structure
as a logical-level simulation of the source QASM (the graphlike
distance check skips itself above its detector cap).

In [ ]:
report = verify(out)
print(report)

## Measure the logical error rate

Uniform circuit-level noise at $p = 10^{-3}$, decoded with
PyMatching (MWPF fallback), capped at $10^{4}$ shots here to keep the
notebook quick.  The LER is program level: a shot fails when any of the
program's observables decodes incorrectly.

In [ ]:
ler_stats = measure_ler(out, p=1e-3, max_shots=10_000)
print(f"LER = {ler_stats.logical_error_rate:.3f}"
      f"  ({ler_stats.errors} failures / {ler_stats.shots} shots)")